In [ ]:
import os
import time
import random
from math import floor

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from matplotlib import pyplot as plt
from tqdm import tqdm
from scipy.signal import wiener

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


In [ ]:
reset_cached_tensors = False

if reset_cached_tensors:
    for cached_tensor in ("./tensor_y.npy", "./tensor_x.npy", "./val_tensor_x.npy", "./val_tensor_y.npy"):
        if os.path.exists(cached_tensor):
            os.remove(cached_tensor)

# Dataset caching: preprocessed DIV2K tensors are stored as .npy files so repeated
# notebook runs can skip image loading and noise generation unless reset_cached_tensors=True.


In [ ]:
# Experiment settings
# DIV2K HR images have variable 2K resolutions. RDUNet downsamples 3 times, so the
# network input must be divisible by 2**3 (=8). A 256x256 DIV2K crop/resized patch
# keeps training with batch_size=150 practical while preserving the DIV2K aspect-free
# denoising task. Increase to 512 only when GPU memory and the 30-hour budget allow it.
epochs = 25
batch_size = 150
lr = 1e-3
max_train_hours = 29.5
early_stop_patience = 5
checkpoint_every_epochs = 5
checkpoint_dir = 'rdunet_checkpoints'
resume_training = True

alpha = 0.1

patch_size = 256
resize_height = patch_size
resize_width = patch_size

ratio = 4
train_val_split_perc = 0.9  # 10 percent for validation from train
val_test_split_perc = 0.5   # 50 percent for test from validation
rdunet_base_filters = 16    # smaller RDUNet width so batch_size=150 fits typical Kaggle GPUs
seed = 42

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [ ]:
class Generator(keras.utils.Sequence):
    def __init__(self, x: list, y: list):
        self.x = x
        self.y = y

    def __len__(self):
        return int(self.x.shape[0])

    def __getitem__(self, index):
        n_samples = self.__len__()
        if n_samples == 0:
            raise IndexError(
                "Generator is empty. Check that the DIV2K dataset is attached and that "
                "img_paths/val_img_paths are not empty before preprocessing."
            )
        if index < 0:
            index = n_samples + index
        if index < 0 or index >= n_samples:
            raise IndexError(
                f"Sample index {index} is out of range for {n_samples} samples. "
                f"Use an index from 0 to {n_samples - 1}."
            )
        x = self.x[index]
        y = self.y[index]
        return x, y


In [ ]:
def add_gaussian_noise(image, sigma=None):
    if sigma is None:
        sigma = np.random.uniform(0.02, 0.08)
    noise = np.random.normal(0, sigma, image.shape)
    noisy = image + noise
    return np.clip(noisy, 0, 1)


def add_realistic_stripe_noise(image, stripe_probability=None, intensity=None):
    if stripe_probability is None:
        stripe_probability = np.random.uniform(0.01, 0.058)
    if intensity is None:
        intensity = np.random.uniform(0.051, 0.23)

    noisy = image.copy()
    h, w, _ = image.shape
    n_cols = max(1, int(w * stripe_probability))
    selected_cols = np.random.choice(w, n_cols, replace=False)

    for col in selected_cols:
        stripe_offset = np.random.normal(0, intensity)
        noisy[:, col, :] += stripe_offset
    return np.clip(noisy, 0, 1)


def add_poisson_noise(image):
    image = np.nan_to_num(image)
    image = np.clip(image, 0, 1)
    noisy = np.random.poisson(image * 255.0) / 255.0
    return np.clip(noisy, 0, 1)


def add_noise(image):
    stripe = add_realistic_stripe_noise(image)
    poisson = add_poisson_noise(image)
    gaussian = add_gaussian_noise(image)
    noisy = (stripe + poisson + gaussian) / 3.0
    return np.clip(noisy, 0, 1)


In [ ]:
def preprocessing(path, ratio, resize_height, resize_width):
    y = tf.keras.utils.load_img(path)
    y = tf.keras.utils.img_to_array(y)
    y = tf.image.resize(
        y,
        [resize_height, resize_width],
        'bicubic',
        antialias=True,
    )
    y = (y / 255.0).numpy().astype(np.float32)
    x = add_noise(y).astype(np.float32)
    return x, y


In [ ]:
def collect_image_paths(*candidate_dirs):
    image_paths = []
    image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
    for candidate_dir in candidate_dirs:
        if not os.path.exists(candidate_dir):
            continue
        for dirname, _, filenames in os.walk(candidate_dir):
            for filename in filenames:
                if filename.lower().endswith(image_extensions):
                    image_paths.append(os.path.join(dirname, filename))
    return sorted(image_paths)


train_dirs = [
    '../input/div2k-dataset/DIV2K_train_HR',
    '/kaggle/input/div2k-dataset/DIV2K_train_HR',
]
valid_dirs = [
    '../input/div2k-dataset/DIV2K_valid_HR',
    '/kaggle/input/div2k-dataset/DIV2K_valid_HR',
]

img_paths = collect_image_paths(*train_dirs)
val_img_paths = collect_image_paths(*valid_dirs)

if len(val_img_paths) == 0 and len(img_paths) > 0:
    val_img_paths = img_paths[floor(len(img_paths) * train_val_split_perc):]
    img_paths = img_paths[:floor(len(img_paths) * train_val_split_perc)]

if len(img_paths) == 0:
    raise FileNotFoundError(
        "No DIV2K training images were found. In Kaggle, use Add Input to attach "
        "the DIV2K dataset and verify that DIV2K_train_HR exists under /kaggle/input/div2k-dataset/."
    )
if len(val_img_paths) == 0:
    raise FileNotFoundError(
        "No DIV2K validation images were found. Attach DIV2K_valid_HR or provide enough "
        "training images to create a validation split."
    )

print('Training images:', len(img_paths))
print('Validation/test images:', len(val_img_paths))
print('First training image:', img_paths[0])


In [ ]:
if not (os.path.exists('./tensor_x.npy')) or not (os.path.exists('./tensor_y.npy')):

    img_lr = []
    img_hr = []

    for i in tqdm(range(len(img_paths))):  #tqdm gives progress bar in display
        x, y = preprocessing(img_paths[i], ratio, resize_height, resize_width)
        img_lr.append(x)
        img_hr.append(y)
    
    tensor_x = tf.convert_to_tensor(img_lr).numpy()
    tensor_y = tf.convert_to_tensor(img_hr).numpy()
    tensor_x.shape

    np.save('./tensor_x.npy', tensor_x)
    np.save('./tensor_y.npy', tensor_y)  
    img_lr = tensor_x
    img_hr = tensor_y
else:
    img_lr = np.load('./tensor_x.npy')
    img_hr = np.load('./tensor_y.npy')


In [ ]:
print('Training tensor shape:', img_lr.shape)
print('Target tensor shape:', img_hr.shape)

if img_lr.shape[0] == 0 or img_hr.shape[0] == 0:
    raise ValueError(
        "Training tensors are empty. Delete tensor_x.npy/tensor_y.npy, set "
        "reset_cached_tensors=True, and rerun after attaching DIV2K."
    )


In [ ]:
if (not os.path.exists('./val_tensor_x.npy')) or (not os.path.exists('./val_tensor_y.npy')):
#check if cached tensors dont exist then create them or load them 
    val_img_lr = []
    val_img_hr = []

    for i in tqdm(range(len(val_img_paths))):
        x, y = preprocessing(
            val_img_paths[i],
            ratio,
            resize_height ,
            resize_width 
        )

        val_img_lr.append(x)
        val_img_hr.append(y)

    val_img_lr = tf.convert_to_tensor(val_img_lr).numpy()
    val_img_hr = tf.convert_to_tensor(val_img_hr).numpy()

    np.save('./val_tensor_x.npy', val_img_lr)
    np.save('./val_tensor_y.npy', val_img_hr)

else:
    val_img_lr = np.load('./val_tensor_x.npy')
    val_img_hr = np.load('./val_tensor_y.npy')

print("Validation LR Shape:", val_img_lr.shape)
print("Validation HR Shape:", val_img_hr.shape)

In [ ]:
split_idx = floor(val_img_lr.shape[0] * val_test_split_perc)
if split_idx <= 0 or split_idx >= val_img_lr.shape[0]:
    raise ValueError(
        f"Invalid validation/test split_idx={split_idx} for {val_img_lr.shape[0]} validation images. "
        "Adjust val_test_split_perc or attach more validation images."
    )

train_generator = Generator(img_lr, img_hr)
val_generator = Generator(val_img_lr[:split_idx], val_img_hr[:split_idx])
test_generator = Generator(val_img_lr[split_idx:], val_img_hr[split_idx:])

print('Validation split index:', split_idx)
print('Validation samples:', len(val_generator), 'Test samples:', len(test_generator))


In [ ]:
# Keep the full training generator available for visualization.
train_generator = Generator(img_lr, img_hr)


In [ ]:
def get_safe_sample_index(generator, requested_index=42):
    n_samples = len(generator)
    if n_samples == 0:
        raise IndexError('Cannot visualize a sample because the generator is empty.')
    if requested_index >= n_samples:
        safe_index = n_samples - 1
        print(
            f'Requested index {requested_index}, but only {n_samples} samples are available. '
            f'Using index {safe_index} instead. Remember Python indexing is zero-based.'
        )
        return safe_index
    return requested_index


def show_generator_sample(generator, requested_index=42):
    sample_index = get_safe_sample_index(generator, requested_index)
    x, y = generator[sample_index]

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(np.clip(x, 0, 1))
    plt.title(f'Noisy sample #{sample_index}')
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(np.clip(y, 0, 1))
    plt.title(f'Clean target #{sample_index}')
    plt.axis('off')
    plt.tight_layout()


show_generator_sample(train_generator, requested_index=42)


In [ ]:
# Optional: visualize another valid sample by changing requested_index.
show_generator_sample(train_generator, requested_index=50)


In [ ]:
#Main Model
import torch
import torch.nn as nn


@torch.no_grad()
def init_weights(init_type='xavier'):
    if init_type == 'xavier':
        init = nn.init.xavier_normal_
    elif init_type == 'he':
        init = nn.init.kaiming_normal_
    else:
        init = nn.init.orthogonal_

    def initializer(m):
        classname = m.__class__.__name__
        if classname.find('Conv2d') != -1:
            init(m.weight)
        elif classname.find('BatchNorm') != -1:
            nn.init.normal_(m.weight, 1.0, 0.01)
            nn.init.zeros_(m.bias)

    return initializer


class DownsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DownsampleBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.actv = nn.PReLU(out_channels)

    def forward(self, x):
        return self.actv(self.conv(x))


class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, cat_channels, out_channels):
        super(UpsampleBlock, self).__init__()

        self.conv = nn.Conv2d(in_channels + cat_channels, out_channels, 3, padding=1)
        self.conv_t = nn.ConvTranspose2d(in_channels, in_channels, 2, stride=2)
        self.actv = nn.PReLU(out_channels)
        self.actv_t = nn.PReLU(in_channels)

    def forward(self, x):
        upsample, concat = x
        upsample = self.actv_t(self.conv_t(upsample))
        return self.actv(self.conv(torch.cat([concat, upsample], 1)))


class InputBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(InputBlock, self).__init__()
        self.conv_1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv_2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        self.actv_1 = nn.PReLU(out_channels)
        self.actv_2 = nn.PReLU(out_channels)

    def forward(self, x):
        x = self.actv_1(self.conv_1(x))
        return self.actv_2(self.conv_2(x))


class OutputBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutputBlock, self).__init__()
        self.conv_1 = nn.Conv2d(in_channels, in_channels, 3, padding=1)
        self.conv_2 = nn.Conv2d(in_channels, out_channels, 3, padding=1)

        self.actv_1 = nn.PReLU(in_channels)
        self.actv_2 = nn.PReLU(out_channels)

    def forward(self, x):
        x = self.actv_1(self.conv_1(x))
        return self.actv_2(self.conv_2(x))


class DenoisingBlock(nn.Module):
    def __init__(self, in_channels, inner_channels, out_channels):
        super(DenoisingBlock, self).__init__()
        self.conv_0 = nn.Conv2d(in_channels, inner_channels, 3, padding=1)
        self.conv_1 = nn.Conv2d(in_channels + inner_channels, inner_channels, 3, padding=1)
        self.conv_2 = nn.Conv2d(in_channels + 2 * inner_channels, inner_channels, 3, padding=1)
        self.conv_3 = nn.Conv2d(in_channels + 3 * inner_channels, out_channels, 3, padding=1)

        self.actv_0 = nn.PReLU(inner_channels)
        self.actv_1 = nn.PReLU(inner_channels)
        self.actv_2 = nn.PReLU(inner_channels)
        self.actv_3 = nn.PReLU(out_channels)

    def forward(self, x):
        out_0 = self.actv_0(self.conv_0(x))

        out_0 = torch.cat([x, out_0], 1)
        out_1 = self.actv_1(self.conv_1(out_0))

        out_1 = torch.cat([out_0, out_1], 1)
        out_2 = self.actv_2(self.conv_2(out_1))

        out_2 = torch.cat([out_1, out_2], 1)
        out_3 = self.actv_3(self.conv_3(out_2))

        return out_3 + x


class RDUNet(nn.Module):
    r"""
    Residual-Dense U-net for image denoising.
    """
    def __init__(self, **kwargs):
        super().__init__()

        channels = kwargs['channels']
        filters_0 = kwargs['base filters']
        filters_1 = 2 * filters_0
        filters_2 = 4 * filters_0
        filters_3 = 8 * filters_0

        # Encoder:
        # Level 0:
        self.input_block = InputBlock(channels, filters_0)
        self.block_0_0 = DenoisingBlock(filters_0, filters_0 // 2, filters_0)
        self.block_0_1 = DenoisingBlock(filters_0, filters_0 // 2, filters_0)
        self.down_0 = DownsampleBlock(filters_0, filters_1)

        # Level 1:
        self.block_1_0 = DenoisingBlock(filters_1, filters_1 // 2, filters_1)
        self.block_1_1 = DenoisingBlock(filters_1, filters_1 // 2, filters_1)
        self.down_1 = DownsampleBlock(filters_1, filters_2)

        # Level 2:
        self.block_2_0 = DenoisingBlock(filters_2, filters_2 // 2, filters_2)
        self.block_2_1 = DenoisingBlock(filters_2, filters_2 // 2, filters_2)
        self.down_2 = DownsampleBlock(filters_2, filters_3)

        # Level 3 (Bottleneck)
        self.block_3_0 = DenoisingBlock(filters_3, filters_3 // 2, filters_3)
        self.block_3_1 = DenoisingBlock(filters_3, filters_3 // 2, filters_3)

        # Decoder
        # Level 2:
        self.up_2 = UpsampleBlock(filters_3, filters_2, filters_2)
        self.block_2_2 = DenoisingBlock(filters_2, filters_2 // 2, filters_2)
        self.block_2_3 = DenoisingBlock(filters_2, filters_2 // 2, filters_2)

        # Level 1:
        self.up_1 = UpsampleBlock(filters_2, filters_1, filters_1)
        self.block_1_2 = DenoisingBlock(filters_1, filters_1 // 2, filters_1)
        self.block_1_3 = DenoisingBlock(filters_1, filters_1 // 2, filters_1)

        # Level 0:
        self.up_0 = UpsampleBlock(filters_1, filters_0, filters_0)
        self.block_0_2 = DenoisingBlock(filters_0, filters_0 // 2, filters_0)
        self.block_0_3 = DenoisingBlock(filters_0, filters_0 // 2, filters_0)

        self.output_block = OutputBlock(filters_0, channels)

    def forward(self, inputs):
        out_0 = self.input_block(inputs)    # Level 0
        out_0 = self.block_0_0(out_0)
        out_0 = self.block_0_1(out_0)

        out_1 = self.down_0(out_0)          # Level 1
        out_1 = self.block_1_0(out_1)
        out_1 = self.block_1_1(out_1)

        out_2 = self.down_1(out_1)          # Level 2
        out_2 = self.block_2_0(out_2)
        out_2 = self.block_2_1(out_2)

        out_3 = self.down_2(out_2)          # Level 3 (Bottleneck)
        out_3 = self.block_3_0(out_3)
        out_3 = self.block_3_1(out_3)

        out_4 = self.up_2([out_3, out_2])   # Level 2
        out_4 = self.block_2_2(out_4)
        out_4 = self.block_2_3(out_4)

        out_5 = self.up_1([out_4, out_1])   # Level 1
        out_5 = self.block_1_2(out_5)
        out_5 = self.block_1_3(out_5)

        out_6 = self.up_0([out_5, out_0])   # Level 0
        out_6 = self.block_0_2(out_6)
        out_6 = self.block_0_3(out_6)

        return self.output_block(out_6) + inputs



In [ ]:
#Metrics for evaluation import torch
from pytorch_msssim import SSIM as _SSIM


class PSNR(object):
    r"""
    Evaluates the PSNR metric in a tensor.
    It can return a result with different reduction methods.

    Args:
        data_range (int, float): Range of the input images.
        reduction (string): Specifies the reduction to apply to the output:
            ``'none'`` | ``'mean'`` | ``'sum'``. ``'none'``: no reduction will be applied,
            ``'mean'``: the sum of the output will be divided by the number of
            elements in the output, ``'sum'``: the output will be summed.
        eps (float): Epsilon value to avoid division by zero.
    """
    def __init__(self, data_range, reduction='none', eps=1e-8):
        self.data_range = data_range
        self.reduction = reduction
        self.eps = eps

    def __call__(self, outputs, targets):
        with torch.set_grad_enabled(False):
            mse = torch.mean((outputs - targets) ** 2., dim=(1, 2, 3))
            psnr = 10. * torch.log10((self.data_range ** 2.) / (mse + self.eps))

            if self.reduction == 'mean':
                return psnr.mean()
            if self.reduction == 'sum':
                return psnr.sum()

            return psnr


class SSIM(object):
    r"""
    Evaluates the SSIM metric in a tensor.
    It can return a result with different reduction methods.

    Args:
        channels (int): Number of channels of the images.
        data_range (int, float): Range of the input images.
        reduction (string): Specifies the reduction to apply to the output:
            ``'none'`` | ``'mean'`` | ``'sum'``. ``'none'``: no reduction will be applied,
            ``'mean'``: the sum of the output will be divided by the number of
            elements in the output, ``'sum'``: the output will be summed.
    """
    def __init__(self, channels, data_range, reduction='none'):
        self.data_range = data_range
        self.reduction = reduction
        self.ssim_module = _SSIM(data_range=data_range, size_average=False, channel=channels)

    def __call__(self, outputs, targets):
        with torch.set_grad_enabled(False):
            ssim = self.ssim_module(outputs, targets)

            if self.reduction == 'mean':
                return ssim.mean()
            if self.reduction == 'sum':
                return ssim.sum()

            return ssim

## RDUNet versus Wiener denoising comparison

The next cells train RDUNet with `batch_size=150`, apply a classical Wiener filter, and compare both against the same noisy DIV2K test split. Since DIV2K is an image restoration dataset and does not contain object bounding boxes, `mAP50` below is reported as an image-level IoU@0.50 reconstruction score: each restored image and clean target are converted to foreground masks and counted as recovered when IoU is at least 0.50.

### Running on Kaggle after importing this notebook

1. Download `rdunet-vs-weiner-on-div2k (1).ipynb` from GitHub.
2. In Kaggle, choose **Code** -> **New Notebook**, then use **File** -> **Upload Notebook** and select this `.ipynb` file.
3. In the right-side **Add Input** / **Input** panel, attach the DIV2K dataset so these folders exist: `/kaggle/input/div2k-dataset/DIV2K_train_HR` and `/kaggle/input/div2k-dataset/DIV2K_valid_HR`.
4. In the right-side notebook **Settings**, set **Accelerator** to a GPU option such as T4/P100. If GPU is disabled, verify the Kaggle account and GPU quota.
5. Run all cells from top to bottom. The first run creates cached `.npy` tensors; later reruns keep those tensors unless `reset_cached_tensors = True`.

### Runtime expectation

Training time depends on the GPU and cache state. The configuration (`batch_size=150`, 256x256 patches, `rdunet_base_filters=16`, early stopping, and `max_train_hours=29.5`) is designed to stay under 30 hours on a Kaggle GPU. With the standard DIV2K training split, batch size 150 gives roughly six training batches per epoch; cached reruns are faster because preprocessing is skipped. After RDUNet training finishes, the next cell prints the observed hours per epoch and projected full-run time from the current execution.


### Why `IndexError` can happen when previewing samples

`train_generator[index]` uses zero-based Python indexing. If the generator has 42 samples, the last valid index is `41`, so `train_generator[42]` raises an `IndexError`. This can also happen when the DIV2K dataset was not attached correctly and the cached/training tensors are empty. The sample-preview cells below now choose a safe fallback index and print a message when the requested index is larger than the available sample count.


In [ ]:
class NumpyImageDataset(Dataset):
    def __init__(self, noisy_images, clean_images):
        self.noisy_images = noisy_images.astype(np.float32)
        self.clean_images = clean_images.astype(np.float32)

    def __len__(self):
        return self.noisy_images.shape[0]

    def __getitem__(self, idx):
        noisy = torch.from_numpy(self.noisy_images[idx]).permute(2, 0, 1)
        clean = torch.from_numpy(self.clean_images[idx]).permute(2, 0, 1)
        return noisy, clean


def make_loader(noisy_images, clean_images, shuffle=False):
    return DataLoader(
        NumpyImageDataset(noisy_images, clean_images),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = make_loader(img_lr, img_hr, shuffle=True)
val_loader = make_loader(val_img_lr[:split_idx], val_img_hr[:split_idx], shuffle=False)
test_noisy = val_img_lr[split_idx:]
test_clean = val_img_hr[split_idx:]
test_loader = make_loader(test_noisy, test_clean, shuffle=False)

print('Device:', device)
print('Train batches:', len(train_loader), 'Validation batches:', len(val_loader), 'Test batches:', len(test_loader))


In [ ]:
def save_training_checkpoint(epoch, model, optimizer, scheduler, scaler, best_val, patience_counter, history):
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict() if scaler is not None else None,
        'best_val': best_val,
        'patience_counter': patience_counter,
        'history': history,
    }
    latest_path = os.path.join(checkpoint_dir, 'rdunet_latest.pt')
    epoch_path = os.path.join(checkpoint_dir, f'rdunet_epoch_{epoch:03d}.pt')
    torch.save(checkpoint, latest_path)
    torch.save(checkpoint, epoch_path)
    print(f'Saved RDUNet checkpoint: {latest_path} and {epoch_path}')


def train_rdunet_model():
    model = RDUNet(**{'channels': 3, 'base filters': rdunet_base_filters}).to(device)
    model.apply(init_weights('he'))

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
    criterion = nn.L1Loss()
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_val = float('inf')
    best_state = None
    patience_counter = 0
    started = time.time()
    history = []
    start_epoch = 1

    latest_checkpoint = os.path.join(checkpoint_dir, 'rdunet_latest.pt')
    if resume_training and os.path.exists(latest_checkpoint):
        checkpoint = torch.load(latest_checkpoint, map_location=device)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        if checkpoint.get('scaler_state') is not None:
            scaler.load_state_dict(checkpoint['scaler_state'])
        best_val = checkpoint.get('best_val', best_val)
        patience_counter = checkpoint.get('patience_counter', patience_counter)
        history = checkpoint.get('history', history)
        start_epoch = checkpoint.get('epoch', 0) + 1
        print(f'Resumed RDUNet training from epoch {start_epoch} using {latest_checkpoint}')

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        train_losses = []
        for noisy, clean in tqdm(train_loader, desc=f'RDUNet epoch {epoch}/{epochs}'):
            noisy = noisy.to(device, non_blocking=True)
            clean = clean.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                restored = torch.clamp(model(noisy), 0, 1)
                loss = criterion(restored, clean)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for noisy, clean in val_loader:
                noisy = noisy.to(device, non_blocking=True)
                clean = clean.to(device, non_blocking=True)
                restored = torch.clamp(model(noisy), 0, 1)
                val_losses.append(criterion(restored, clean).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses)) if len(val_losses) else train_loss
        scheduler.step(val_loss)
        elapsed_hours = (time.time() - started) / 3600.0
        history.append({'epoch': epoch, 'train_l1': train_loss, 'val_l1': val_loss, 'elapsed_hours': elapsed_hours})
        print(f'Epoch {epoch}: train_l1={train_loss:.5f}, val_l1={val_loss:.5f}, elapsed_hours={elapsed_hours:.2f}')

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            patience_counter = 0
            torch.save(best_state, 'rdunet_div2k_best.pt')
            print('Saved new best RDUNet weights: rdunet_div2k_best.pt')
        else:
            patience_counter += 1

        if epoch % checkpoint_every_epochs == 0 or epoch == epochs:
            save_training_checkpoint(epoch, model, optimizer, scheduler, scaler, best_val, patience_counter, history)

        if patience_counter >= early_stop_patience:
            print('Early stopping triggered.')
            save_training_checkpoint(epoch, model, optimizer, scheduler, scaler, best_val, patience_counter, history)
            break
        if elapsed_hours >= max_train_hours:
            print(f'Stopping because elapsed training time reached {max_train_hours} hours.')
            save_training_checkpoint(epoch, model, optimizer, scheduler, scaler, best_val, patience_counter, history)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    elif os.path.exists('rdunet_div2k_best.pt'):
        model.load_state_dict(torch.load('rdunet_div2k_best.pt', map_location=device))
    return model, pd.DataFrame(history)


rdunet_model, rdunet_history = train_rdunet_model()
rdunet_history


In [ ]:
def estimate_training_time(history, total_epochs=epochs):
    if history.empty or 'elapsed_hours' not in history:
        print('No RDUNet history is available yet. Run the RDUNet training cell first.')
        return None

    completed_epochs = int(history['epoch'].max())
    elapsed_hours = float(history['elapsed_hours'].iloc[-1])
    hours_per_epoch = elapsed_hours / max(completed_epochs, 1)
    projected_total_hours = min(hours_per_epoch * total_epochs, max_train_hours)
    remaining_hours = max(projected_total_hours - elapsed_hours, 0.0)

    print(f'Completed epochs: {completed_epochs}/{total_epochs}')
    print(f'Elapsed training time: {elapsed_hours:.2f} hours')
    print(f'Observed time per epoch: {hours_per_epoch:.2f} hours')
    print(f'Projected total training time: {projected_total_hours:.2f} hours')
    print(f'Projected remaining time: {remaining_hours:.2f} hours')
    print(f'Training guard: stops at {max_train_hours:.2f} hours even if all epochs are not complete')

    return {
        'completed_epochs': completed_epochs,
        'elapsed_hours': elapsed_hours,
        'hours_per_epoch': hours_per_epoch,
        'projected_total_hours': projected_total_hours,
        'remaining_hours': remaining_hours,
    }


training_time_estimate = estimate_training_time(rdunet_history)


In [ ]:
def apply_wiener_image(image, mysize=5):
    restored_channels = [wiener(image[:, :, channel], mysize=mysize) for channel in range(image.shape[-1])]
    restored = np.stack(restored_channels, axis=-1)
    return np.nan_to_num(np.clip(restored, 0, 1)).astype(np.float32)


def apply_wiener_batch(noisy_images, mysize=5):
    return np.stack([apply_wiener_image(image, mysize=mysize) for image in tqdm(noisy_images, desc='Wiener filtering')])


def predict_rdunet(model, loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for noisy, _ in tqdm(loader, desc='RDUNet inference'):
            noisy = noisy.to(device, non_blocking=True)
            restored = torch.clamp(model(noisy), 0, 1).cpu().permute(0, 2, 3, 1).numpy()
            predictions.append(restored)
    return np.concatenate(predictions, axis=0).astype(np.float32)


wiener_restored = apply_wiener_batch(test_noisy, mysize=5)
rdunet_restored = predict_rdunet(rdunet_model, test_loader)


In [ ]:
def mse_score(pred, target):
    return np.mean((pred - target) ** 2, axis=(1, 2, 3))


def psnr_score(pred, target, eps=1e-8):
    mse = mse_score(pred, target)
    return 10.0 * np.log10(1.0 / (mse + eps))


def map50_reconstruction_score(pred, target, threshold=0.5):
    pred_mask = pred.mean(axis=-1) >= threshold
    target_mask = target.mean(axis=-1) >= threshold
    intersection = np.logical_and(pred_mask, target_mask).sum(axis=(1, 2))
    union = np.logical_or(pred_mask, target_mask).sum(axis=(1, 2))
    iou = np.divide(intersection, np.maximum(union, 1))
    return (iou >= 0.50).astype(np.float32), iou


def summarize_method(name, restored, noisy, clean):
    noisy_mse = mse_score(noisy, clean)
    restored_mse = mse_score(restored, clean)
    recovery_pct = 100.0 * (noisy_mse - restored_mse) / np.maximum(noisy_mse, 1e-8)

    noisy_map50, _ = map50_reconstruction_score(noisy, clean)
    restored_map50, restored_iou = map50_reconstruction_score(restored, clean)
    performance_loss_recovered = 100.0 * (restored_map50.mean() - noisy_map50.mean()) / max(1.0 - noisy_map50.mean(), 1e-8)

    return {
        'method': name,
        'mAP50': restored_map50.mean(),
        'mean_iou': restored_iou.mean(),
        'recovery_percentage_mse': recovery_pct.mean(),
        'performance_loss_recovered_mAP50': performance_loss_recovered,
        'psnr': psnr_score(restored, clean).mean(),
        'mse': restored_mse.mean(),
    }


baseline = summarize_method('Noisy baseline', test_noisy, test_noisy, test_clean)
wiener_metrics = summarize_method('Wiener filter', wiener_restored, test_noisy, test_clean)
rdunet_metrics = summarize_method('RDUNet', rdunet_restored, test_noisy, test_clean)
comparison = pd.DataFrame([baseline, wiener_metrics, rdunet_metrics])
comparison


In [ ]:
def show_requested_sample(index=0):
    """Show the same DIV2K test sample in the requested workflow format."""
    workflows = [
        ('1. Wiener filter', [test_clean[index], test_noisy[index], wiener_restored[index]]),
        ('2. RDUNet', [test_clean[index], test_noisy[index], rdunet_restored[index]]),
    ]
    column_titles = ['Clean sample image', 'Noisy image', 'Clean image after filtering']

    plt.figure(figsize=(15, 8))
    for row, (workflow_name, images) in enumerate(workflows):
        for col, (title, image) in enumerate(zip(column_titles, images)):
            axis = plt.subplot(2, 3, row * 3 + col + 1)
            axis.imshow(np.clip(image, 0, 1))
            axis.set_title(f'{workflow_name}\n{title}')
            axis.axis('off')
    plt.tight_layout()


show_requested_sample(index=0)
